In [1]:
import os
import random
import tempfile
from typing import Optional, Dict, Any, List, Tuple

import boto3
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ============================================================
# Data utilities
# ============================================================

def make_stratified_splits_first_n(
    h5_path: str,
    max_samples: int = 10_000,
    snr_bin_db: Optional[int] = 2,
    train_frac: float = 0.70,
    val_frac: float = 0.15,
    test_frac: float = 0.15,
    seed: int = 42,
):
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-9

    with h5py.File(h5_path, "r") as f:
        n_total = f["Y"].shape[0]
        n = min(max_samples, n_total)
        Y = f["Y"][:n]
        Z = f["Z"][:n]

    cls = np.argmax(Y, axis=1).astype(int)
    snr = np.asarray(Z).squeeze()

    if snr_bin_db is not None:
        snr_b = (np.round(snr / snr_bin_db) * snr_bin_db).astype(int)
    else:
        snr_b = snr.astype(int)

    strata = np.array([f"{c}_{s}" for c, s in zip(cls, snr_b)], dtype=object)
    idx = np.arange(n)

    temp_frac = 1.0 - train_frac
    train_idx, temp_idx = train_test_split(
        idx, test_size=temp_frac, stratify=strata, random_state=seed
    )

    val_within_temp = val_frac / (val_frac + test_frac)
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=(1.0 - val_within_temp),
        stratify=strata[temp_idx],
        random_state=seed,
    )

    return train_idx, val_idx, test_idx


def load_labels_and_snr_first_n(h5_path: str, max_samples: int = 10_000):
    with h5py.File(h5_path, "r") as f:
        n_total = f["Y"].shape[0]
        n = min(max_samples, n_total)
        Y = f["Y"][:n]
        Z = f["Z"][:n]
    y = np.argmax(Y, axis=1).astype(int)
    z = np.asarray(Z).squeeze().astype(float)
    return y, z


# ============================================================
# Dataset
# ============================================================

class RadioMLH5Dataset(Dataset):
    """
    HDF5-backed dataset. Safe for notebooks/Windows with num_workers=0.
    Returns:
      x: [2, L] float32  (I,Q)
      y: scalar class index
    """
    def __init__(self, h5_path: str, indices: np.ndarray):
        self.h5_path = str(h5_path)
        self.indices = np.asarray(indices, dtype=np.int64)
        self._file = None

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        if self._file is None:
            self._file = h5py.File(self.h5_path, "r")

        idx = int(self.indices[i])
        X = np.asarray(self._file["X"][idx])
        Y = np.asarray(self._file["Y"][idx])

        if X.ndim != 2:
            raise ValueError(f"Unexpected X shape {X.shape} at idx={idx}")

        if X.shape[-1] == 2:
            x2l = X.T
        elif X.shape[0] == 2:
            x2l = X
        else:
            raise ValueError(f"Can't interpret IQ axes for X shape {X.shape} at idx={idx}")

        y = int(np.argmax(Y))
        return torch.tensor(x2l, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


def make_loader(h5_path, indices, batch_size, num_workers, shuffle):
    ds = RadioMLH5Dataset(h5_path, indices)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=(shuffle and len(indices) >= batch_size),
    )


# ============================================================
# Curriculum helpers
# ============================================================

def snr_stage_mask(
    snr_values: np.ndarray,
    stage_name: str,
    high_thr: float = 10.0,
    low_thr: float = 0.0,
):
    if stage_name == "high":
        return snr_values >= high_thr
    elif stage_name == "high_med":
        return snr_values >= low_thr
    elif stage_name == "all":
        return np.ones_like(snr_values, dtype=bool)
    else:
        raise ValueError(f"Unknown stage_name={stage_name}")


def build_curriculum_schedule(stage_epochs: List[int]) -> List[str]:
    """
    stage_epochs = [n_high, n_high_med, n_all]
    Example: [5, 8, 12]
    """
    if len(stage_epochs) != 3:
        raise ValueError("stage_epochs must have length 3: [n_high, n_high_med, n_all]")
    if any(e < 0 for e in stage_epochs):
        raise ValueError("stage_epochs entries must be >= 0")

    return (
        ["high"] * stage_epochs[0]
        + ["high_med"] * stage_epochs[1]
        + ["all"] * stage_epochs[2]
    )


# ============================================================
# Preprocessing + ResNet
# ============================================================

class IQSpectrogramPreprocessor(nn.Module):
    """
    [B,2,L] -> remove DC -> robust RMS normalize -> complex STFT -> log-mag spectrogram [B,1,F,T]
    """
    def __init__(
        self,
        n_fft: int = 256,
        hop_length: int = 64,
        win_length: int = 256,
        eps: float = 1e-10,
        robust_clip_k: float = 6.0,
        rms_floor: float = 1e-6,
    ):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.eps = eps
        self.robust_clip_k = robust_clip_k
        self.rms_floor = rms_floor
        self.register_buffer("window", torch.hann_window(win_length), persistent=False)

    def _remove_dc(self, x_complex: torch.Tensor) -> torch.Tensor:
        return x_complex - x_complex.mean(dim=1, keepdim=True)

    def _robust_rms_normalize(self, x_complex: torch.Tensor) -> torch.Tensor:
        r = torch.abs(x_complex)
        med = r.median(dim=1, keepdim=True).values
        mad = (r - med).abs().median(dim=1, keepdim=True).values
        mad = torch.clamp(mad, min=self.rms_floor)

        clip = self.robust_clip_k * mad
        scale = torch.minimum(torch.ones_like(r), clip / torch.clamp(r, min=self.rms_floor))
        x_clipped = x_complex * scale

        rms = torch.sqrt(torch.mean(torch.abs(x_clipped) ** 2, dim=1, keepdim=True))
        rms = torch.clamp(rms, min=self.rms_floor)
        return x_clipped / rms

    def forward(self, x_iq: torch.Tensor) -> torch.Tensor:
        x_complex = torch.complex(x_iq[:, 0, :], x_iq[:, 1, :])
        x_complex = self._remove_dc(x_complex)
        x_complex = self._robust_rms_normalize(x_complex)

        Z = torch.stft(
            x_complex,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=self.window,
            center=False,
            return_complex=True,
            onesided=False,
        )  # [B,F,T]

        S = torch.log(torch.abs(Z) + self.eps)
        S = torch.fft.fftshift(S, dim=1)
        return S.unsqueeze(1)  # [B,1,F,T]


class ResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1, dropout_p: float = 0.10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.drop1 = nn.Dropout2d(dropout_p) if dropout_p > 0 else nn.Identity()

        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.drop2 = nn.Dropout2d(dropout_p) if dropout_p > 0 else nn.Identity()

        if stride != 1 or in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.proj = nn.Identity()

    def forward(self, x):
        identity = self.proj(x)

        y = self.conv1(x)
        y = self.bn1(y)
        y = self.act(y)
        y = self.drop1(y)

        y = self.conv2(y)
        y = self.bn2(y)
        y = self.drop2(y)

        y = y + identity
        y = self.act(y)
        return y


class ResNetSpectrogramClassifier(nn.Module):
    """
    STFT front-end + compact ResNet classifier.
    """
    def __init__(
        self,
        num_classes: int,
        n_fft: int = 256,
        hop_length: int = 64,
        win_length: int = 256,
        eps: float = 1e-10,
        base_ch: int = 32,
        dropout_p: float = 0.10,
        robust_clip_k: float = 6.0,
        rms_floor: float = 1e-6,
    ):
        super().__init__()
        self.pre = IQSpectrogramPreprocessor(
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            eps=eps,
            robust_clip_k=robust_clip_k,
            rms_floor=rms_floor,
        )

        self.stem = nn.Sequential(
            nn.Conv2d(1, base_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True),
        )

        self.layer1 = nn.Sequential(
            ResBlock(base_ch, base_ch, stride=1, dropout_p=dropout_p),
            ResBlock(base_ch, base_ch, stride=1, dropout_p=dropout_p),
        )
        self.layer2 = nn.Sequential(
            ResBlock(base_ch, base_ch * 2, stride=2, dropout_p=dropout_p),
            ResBlock(base_ch * 2, base_ch * 2, stride=1, dropout_p=dropout_p),
        )
        self.layer3 = nn.Sequential(
            ResBlock(base_ch * 2, base_ch * 4, stride=2, dropout_p=dropout_p),
            ResBlock(base_ch * 4, base_ch * 4, stride=1, dropout_p=dropout_p),
        )
        self.layer4 = nn.Sequential(
            ResBlock(base_ch * 4, base_ch * 8, stride=2, dropout_p=dropout_p),
            ResBlock(base_ch * 8, base_ch * 8, stride=1, dropout_p=dropout_p),
        )

        self.head = nn.Linear(base_ch * 8, num_classes)

    def forward(self, x_iq: torch.Tensor) -> torch.Tensor:
        x = self.pre(x_iq)
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = x.mean(dim=(2, 3))
        return self.head(x)


# ============================================================
# Callbacks + checkpointing
# ============================================================

class EarlyStopping:
    def __init__(self, patience: int = 6, min_delta: float = 1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = float("inf")
        self.bad_epochs = 0

    def step(self, val_loss: float) -> bool:
        if val_loss < self.best - self.min_delta:
            self.best = val_loss
            self.bad_epochs = 0
            return False
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience


class StageAwareModelCheckpoint:
    """
    Saves best-by-monitored-loss. Stage name is embedded in chkpt_path.
    """
    def __init__(self, chkpt_dir: str, base_name: str = "best_val_fixed_loss"):
        self.chkpt_dir = chkpt_dir
        self.base_name = base_name
        self.best = float("inf")
        self.best_path = None

    def make_path(self, stage_name: str) -> str:
        return os.path.join(self.chkpt_dir, f"{self.base_name}_{stage_name}.pt")

    def step(self, monitored_loss: float, model: nn.Module, stage_name: str, extra: Dict[str, Any]) -> bool:
        if monitored_loss < self.best:
            self.best = monitored_loss
            os.makedirs(self.chkpt_dir, exist_ok=True)
            chkpt_path = self.make_path(stage_name)
            torch.save({"model_state": model.state_dict(), **extra}, chkpt_path)
            self.best_path = chkpt_path
            return True
        return False


# ============================================================
# Eval
# ============================================================

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction="sum")

        total_loss += float(loss.item())
        total_correct += int((logits.argmax(dim=1) == y).sum().item())
        total += y.numel()

    return total_loss / max(total, 1), total_correct / max(total, 1)


# ============================================================
# AWS helpers
# ============================================================

def upload_file_to_s3(local_path: str, s3_uri: str):
    """
    s3_uri format: s3://bucket/key
    """
    if not s3_uri.startswith("s3://"):
        raise ValueError("s3_uri must start with s3://")

    no_scheme = s3_uri[len("s3://"):]
    bucket, key = no_scheme.split("/", 1)

    s3 = boto3.client("s3")
    s3.upload_file(local_path, bucket, key)
    return s3_uri


def shutdown_sagemaker_jupyterlab_app(
    domain_id: str,
    user_profile_name: str,
    app_name: str = "default",
):
    """
    Stops/deletes a SageMaker Studio JupyterLab app via DeleteApp.
    """
    sm = boto3.client("sagemaker")
    return sm.delete_app(
        DomainId=domain_id,
        UserProfileName=user_profile_name,
        AppType="JupyterLab",
        AppName=app_name,
    )


# ============================================================
# Training
# ============================================================

def train_resnet_curriculum_dual_val(
    h5_path: str,
    log_dir: str = "./runs/rml2018_resnet_curriculum_dualval",
    stage_epochs: List[int] = [5, 5, 10],
    batch_size: int = 2560,
    lr: float = 1e-3,
    num_workers: int = 0,
    seed: int = 42,
    early_patience: int = 6,
    max_samples: int = 10_000,
    dropout_p: float = 0.10,
    high_thr: float = 10.0,
    low_thr: float = 0.0,
    save_best_to_s3: bool = False,
    best_model_s3_uri: Optional[str] = None,
    shutdown_sagemaker_jupyterlab: bool = False,
    sagemaker_domain_id: Optional[str] = None,
    sagemaker_user_profile_name: Optional[str] = None,
    sagemaker_jupyterlab_app_name: str = "default",
):
    """
    stage_epochs example:
      [3, 4, 8] means
        3 epochs of 'high'
        4 epochs of 'high_med'
        8 epochs of 'all'
    """
    seed_everything(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(log_dir, exist_ok=True)

    schedule = build_curriculum_schedule(stage_epochs)
    total_epochs = len(schedule)

    with h5py.File(h5_path, "r") as f:
        num_classes = int(f["Y"].shape[1])

    train_idx_full, val_idx_full, test_idx = make_stratified_splits_first_n(
        h5_path=h5_path,
        max_samples=max_samples,
        snr_bin_db=2,
        train_frac=0.70,
        val_frac=0.15,
        test_frac=0.15,
        seed=seed,
    )

    _, snr_all = load_labels_and_snr_first_n(h5_path, max_samples=max_samples)

    val_loader_fixed = make_loader(
        h5_path=h5_path,
        indices=val_idx_full,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,
    )
    test_loader = make_loader(
        h5_path=h5_path,
        indices=test_idx,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,
    )

    model = ResNetSpectrogramClassifier(
        num_classes=num_classes,
        n_fft=256,
        hop_length=64,
        win_length=256,
        base_ch=32,
        dropout_p=dropout_p,
        robust_clip_k=6.0,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    lr_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2, verbose=True
    )

    writer = SummaryWriter(log_dir=log_dir)
    checkpoint = StageAwareModelCheckpoint(
        chkpt_dir=os.path.join(log_dir, "checkpoints"),
        base_name="best_val_fixed_loss",
    )
    early_stop = EarlyStopping(patience=early_patience, min_delta=1e-4)

    global_step = 0
    best_epoch = -1

    epoch_pbar = tqdm(range(1, total_epochs + 1), desc=f"Epochs (dual val, first {max_samples})", leave=True)

    try:
        for epoch in epoch_pbar:
            stage = schedule[epoch - 1]

            train_snrs = snr_all[train_idx_full]
            val_snrs = snr_all[val_idx_full]

            train_mask = snr_stage_mask(train_snrs, stage_name=stage, high_thr=high_thr, low_thr=low_thr)
            val_mask_stage = snr_stage_mask(val_snrs, stage_name=stage, high_thr=high_thr, low_thr=low_thr)

            train_idx_stage = train_idx_full[train_mask]
            val_idx_stage = val_idx_full[val_mask_stage]

            if len(train_idx_stage) == 0:
                raise RuntimeError(f"No training samples left for stage '{stage}'")
            if len(val_idx_stage) == 0:
                raise RuntimeError(f"No stage validation samples left for stage '{stage}'")

            train_loader = make_loader(
                h5_path=h5_path,
                indices=train_idx_stage,
                batch_size=batch_size,
                num_workers=num_workers,
                shuffle=True,
            )
            val_loader_stage = make_loader(
                h5_path=h5_path,
                indices=val_idx_stage,
                batch_size=batch_size,
                num_workers=num_workers,
                shuffle=False,
            )

            model.train()
            running_loss_sum = 0.0
            running_count = 0
            ema_loss = None
            ema_alpha = 0.05

            batch_pbar = tqdm(
                train_loader,
                desc=f"Train (epoch {epoch}/{total_epochs}, stage={stage}, train_n={len(train_idx_stage)}, val_stage_n={len(val_idx_stage)})",
                leave=False,
            )

            for x, y in batch_pbar:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                loss = F.cross_entropy(logits, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

                bs = y.numel()
                running_loss_sum += float(loss.item()) * bs
                running_count += bs

                if ema_loss is None:
                    ema_loss = float(loss.item())
                else:
                    ema_loss = (1 - ema_alpha) * ema_loss + ema_alpha * float(loss.item())

                batch_pbar.set_postfix({
                    "stage": stage,
                    "loss": f"{float(loss.item()):.4f}",
                    "ema": f"{ema_loss:.4f}",
                    "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
                })

                writer.add_scalar("train/loss_step", float(loss.item()), global_step)
                writer.add_scalar("train/lr", float(optimizer.param_groups[0]["lr"]), global_step)
                global_step += 1

            train_loss = running_loss_sum / max(running_count, 1)

            val_stage_loss, val_stage_acc = evaluate(model, val_loader_stage, device)
            val_fixed_loss, val_fixed_acc = evaluate(model, val_loader_fixed, device)

            lr_sched.step(val_fixed_loss)

            writer.add_scalar("train/loss_epoch", train_loss, epoch)
            writer.add_scalar("val_stage/loss", val_stage_loss, epoch)
            writer.add_scalar("val_stage/accuracy", val_stage_acc, epoch)
            writer.add_scalar("val_fixed/loss", val_fixed_loss, epoch)
            writer.add_scalar("val_fixed/accuracy", val_fixed_acc, epoch)
            writer.add_text(
                "curriculum/stage",
                f"epoch={epoch}, stage={stage}, train_n={len(train_idx_stage)}, "
                f"val_stage_n={len(val_idx_stage)}, val_fixed_n={len(val_idx_full)}",
                epoch,
            )

            epoch_pbar.set_postfix({
                "stage": stage,
                "train_loss": f"{train_loss:.4f}",
                "val_stage_loss": f"{val_stage_loss:.4f}",
                "val_fixed_loss": f"{val_fixed_loss:.4f}",
                "val_fixed_acc": f"{val_fixed_acc:.4f}",
            })

            saved = checkpoint.step(
                monitored_loss=val_fixed_loss,
                model=model,
                stage_name=stage,
                extra={
                    "epoch": epoch,
                    "val_stage_loss": float(val_stage_loss),
                    "val_stage_acc": float(val_stage_acc),
                    "val_fixed_loss": float(val_fixed_loss),
                    "val_fixed_acc": float(val_fixed_acc),
                    "dropout_p": float(dropout_p),
                    "curriculum_stage": stage,
                    "stage_epochs": list(stage_epochs),
                }
            )
            if saved:
                best_epoch = epoch
                print(
                    f"\nSaved new best checkpoint at epoch {epoch} "
                    f"(stage={stage}, val_fixed_loss={val_fixed_loss:.4f}) "
                    f"-> {checkpoint.best_path}"
                )

            if early_stop.step(val_fixed_loss):
                print(
                    f"\nEarly stopping at epoch {epoch} "
                    f"(best epoch {best_epoch}, best val_fixed_loss {checkpoint.best:.4f})"
                )
                break

        if checkpoint.best_path is not None and os.path.exists(checkpoint.best_path):
            ckpt = torch.load(checkpoint.best_path, map_location=device)
            model.load_state_dict(ckpt["model_state"])
            print(
                f"\nLoaded best model from epoch {ckpt.get('epoch')} "
                f"(stage={ckpt.get('curriculum_stage')}) "
                f"with val_fixed_loss={ckpt.get('val_fixed_loss'):.4f}"
            )

        test_loss, test_acc = evaluate(model, test_loader, device)
        writer.add_scalar("test/loss", test_loss, 0)
        writer.add_scalar("test/accuracy", test_acc, 0)

        print(f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")
        print(f"TensorBoard logs: {log_dir}")
        print(f"Best checkpoint path: {checkpoint.best_path}")

        # Optional S3 upload
        if save_best_to_s3:
            if checkpoint.best_path is None:
                raise RuntimeError("No best checkpoint available to upload.")
            if not best_model_s3_uri:
                raise ValueError("best_model_s3_uri must be set when save_best_to_s3=True")
            uploaded_uri = upload_file_to_s3(checkpoint.best_path, best_model_s3_uri)
            print(f"Uploaded best checkpoint to {uploaded_uri}")

    finally:
        writer.close()

        # Optional SageMaker JupyterLab shutdown
        # Do this absolutely last.
        if shutdown_sagemaker_jupyterlab:
            if not sagemaker_domain_id or not sagemaker_user_profile_name:
                raise ValueError(
                    "sagemaker_domain_id and sagemaker_user_profile_name are required "
                    "when shutdown_sagemaker_jupyterlab=True"
                )
            print(
                "Requesting SageMaker JupyterLab shutdown. "
                "If successful, this notebook session will terminate."
            )
            shutdown_sagemaker_jupyterlab_app(
                domain_id=sagemaker_domain_id,
                user_profile_name=sagemaker_user_profile_name,
                app_name=sagemaker_jupyterlab_app_name,
            )

    return model

2026-03-20 15:22:58.958974: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774020178.975276     345 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774020178.980364     345 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-20 15:22:58.996710: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
H5_PATH = "./GOLD_XYZ_OSC.0001_1024.hdf5"
model = train_resnet_curriculum_dual_val(
    h5_path=H5_PATH,
    batch_size=2000,
    log_dir="./runs/rml2018_resnet_curriculum_dualval",
    stage_epochs=[2, 5, 10],
    lr=1e-3,
    num_workers=4,     # Windows-safe
    seed=42,
    early_patience=15,
    max_samples=1_000_000,
    dropout_p=0.10,
    high_thr=10.0,     # high SNR >= 10 dB
    low_thr=0.0,       # medium+high in stage 2 means SNR >= 0 dB
)

/opt/conda/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epochs (dual val, first 100000):   0%|          | 0/17 [00:00<?, ?it/s]

Train (epoch 1/17, stage=high, train_n=26991, val_stage_n=5785):   0%|          | 0/13 [00:00<?, ?it/s]


Saved new best checkpoint at epoch 1 (stage=high, val_fixed_loss=0.0000) -> ./runs/rml2018_resnet_curriculum_dualval/checkpoints/best_val_fixed_loss_high.pt


Train (epoch 2/17, stage=high, train_n=26991, val_stage_n=5785):   0%|          | 0/13 [00:00<?, ?it/s]

...

Train (epoch 13/17, stage=all, train_n=69999, val_stage_n=15000):   0%|          | 0/34 [00:00<?, ?it/s]

Train (epoch 14/17, stage=all, train_n=69999, val_stage_n=15000):   0%|          | 0/34 [00:00<?, ?it/s]

Train (epoch 15/17, stage=all, train_n=69999, val_stage_n=15000):   0%|          | 0/34 [00:00<?, ?it/s]

Train (epoch 16/17, stage=all, train_n=69999, val_stage_n=15000):   0%|          | 0/34 [00:00<?, ?it/s]


Early stopping at epoch 16 (best epoch 2, best val_fixed_loss 0.0000)

Loaded best model from epoch 2 (stage=high) with val_fixed_loss=0.0000
Test loss: 0.0000 | Test acc: 1.0000
TensorBoard logs: ./runs/rml2018_resnet_curriculum_dualval
Best checkpoint path: ./runs/rml2018_resnet_curriculum_dualval/checkpoints/best_val_fixed_loss_high.pt


In [6]:
H5_PATH = "./GOLD_XYZ_OSC.0001_1024.hdf5"
model = train_resnet_curriculum_dual_val(
    h5_path=H5_PATH,
    batch_size=2000,
    log_dir="./runs/rml2018_resnet_curriculum_dualval_1e6",
    stage_epochs=[2, 8, 10],
    lr=1e-3,
    num_workers=4,     # Windows-safe
    seed=42,
    early_patience=15,
    max_samples=1_000_000,
    dropout_p=0.10,
    high_thr=1.0,     # high SNR >= 10 dB
    low_thr=-8.0,       # medium+high in stage 2 means SNR >= 0 dB
)

/opt/conda/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epochs (dual val, first 1000000):   0%|          | 0/20 [00:00<?, ?it/s]

Train (epoch 1/20, stage=high, train_n=387074, val_stage_n=82945):   0%|          | 0/193 [00:00<?, ?it/s]


Saved new best checkpoint at epoch 1 (stage=high, val_fixed_loss=5.1719) -> ./runs/rml2018_resnet_curriculum_dualval_1e6/checkpoints/best_val_fixed_loss_high.pt


Train (epoch 2/20, stage=high, train_n=387074, val_stage_n=82945):   0%|          | 0/193 [00:00<?, ?it/s]

...


Saved new best checkpoint at epoch 5 (stage=high_med, val_fixed_loss=1.5812) -> ./runs/rml2018_resnet_curriculum_dualval_1e6/checkpoints/best_val_fixed_loss_high_med.pt


Train (epoch 6/20, stage=high_med, train_n=527972, val_stage_n=113140):   0%|          | 0/263 [00:00<?, ?it/s…


Saved new best checkpoint at epoch 6 (stage=high_med, val_fixed_loss=1.5553) -> ./runs/rml2018_resnet_curriculum_dualval_1e6/checkpoints/best_val_fixed_loss_high_med.pt


Train (epoch 7/20, stage=high_med, train_n=527972, val_stage_n=113140):   0%|          | 0/263 [00:00<?, ?it/s…

KeyboardInterrupt: 